<a href="https://colab.research.google.com/github/ttienthuy131/climate-finance-research/blob/main/wsj_fetcher_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WSJ Article Fetcher v2 — Scalable Two-Phase Pipeline

Reliably fetches ~26K+ paywalled WSJ articles across one or more Colab sessions.

**Phase 1: URL Discovery** (no authentication needed)
- Crawls WSJ monthly archive sitemaps (2013–2026)
- Filters by oil/energy/climate keywords in URL slugs
- Saves URL list to Google Drive (survives runtime restarts)

**Phase 2: Article Fetching** (authenticated, resumable)
- **Single-threaded** with human-like random delays (1–3s) to avoid bot detection
- Auto-login with **manual cookie injection** fallback (bypasses CAPTCHA)
- Checkpoints every 10 articles to Google Drive — never repeats work
- URLs are shuffled to avoid sequential-access bot signatures
- On session expiry (~25 min): auto re-login and continue seamlessly
- On IP block: save progress, restart Colab runtime for new IP, resume

**Timeline:** ~26K articles × ~2s avg = ~14h. Split across 2 free-tier Colab sessions comfortably. Or finish in one session if delays are kept short (~1s avg → ~7h).

In [ ]:
!pip install -q trafilatura curl_cffi pandas==2.2.2 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 10.5 MB/s eta 0:00:00


In [ ]:
import base64
import json
import os
import random
import re
import time
from datetime import datetime
from getpass import getpass
from urllib.parse import urlparse, parse_qs
from xml.etree import ElementTree as ET

import pandas as pd
import requests
import trafilatura
from curl_cffi import requests as cffi_requests

In [ ]:
# ── Configuration ─────────────────────────────────────────────

KEYWORDS = [
    # Middle East
    "middle east", "gulf", "persian gulf", "strait of hormuz",
    "saudi arabia", "iran", "iraq", "uae", "qatar", "kuwait", "oman",
    # Energy
    "oil", "crude oil", "brent", "opec", "opec+",
    "energy market", "energy supply", "petroleum", "natural gas",
    # Energy security
    "supply disruption", "oil supply", "pipeline",
    "oil tanker", "shipping route", "production cut",
    "sanctions", "energy crisis",
    # Climate / transition
    "energy transition", "renewable energy",
    "solar power", "wind power",
    "climate policy", "carbon emissions",
    "net zero", "decarbonization",
]

CONFIG = {
    "keywords": KEYWORDS,
    "start_date": "2013-01-01",
    "end_date": "2026-03-15",
    # Fetching — single-threaded with human-like delays
    "delay_range": (1.0, 3.0),       # Random delay between requests (seconds)
    "checkpoint_every": 10,           # Flush to disk every N articles
    # Session management
    "session_ttl_minutes": 20,        # Re-login before ~25-min hard expiry
    "max_auto_logins": 4,             # Switch to manual cookies after this many
}

print(f"Keywords: {len(KEYWORDS)}")
print(f"Date range: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"Delay: {CONFIG['delay_range'][0]}-{CONFIG['delay_range'][1]}s per request")

Keywords: 36
Date range: 2013-01-01 to 2026-03-15
Delay: 1.0-3.0s per request


In [ ]:
# ── Google Drive mount (persistence across runtime restarts) ──

USE_DRIVE = True  # Set False to use local /content only

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/wsj_data'
else:
    DATA_DIR = '/content/wsj_data'

os.makedirs(DATA_DIR, exist_ok=True)

URL_LIST_PATH   = os.path.join(DATA_DIR, 'wsj_urls.csv')
CHECKPOINT_PATH = os.path.join(DATA_DIR, 'wsj_checkpoint.csv')
OUTPUT_PATH     = os.path.join(DATA_DIR, 'wsj_articles.csv')

print(f"Data dir:   {DATA_DIR}")
print(f"URL list:   {URL_LIST_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output:     {OUTPUT_PATH}")

Mounted at /content/drive
Data dir:   /content/drive/MyDrive/wsj_data
URL list:   /content/drive/MyDrive/wsj_data/wsj_urls.csv
Checkpoint: /content/drive/MyDrive/wsj_data/wsj_checkpoint.csv
Output:     /content/drive/MyDrive/wsj_data/wsj_articles.csv


---
## Phase 1: URL Discovery (no authentication needed)

Crawls WSJ's monthly archive sitemaps and filters by keyword. Results are cached to Google Drive — this cell is idempotent (re-running loads the cache).

In [ ]:
NS = {"s": "http://www.sitemaps.org/schemas/sitemap/0.9"}
SITEMAP_UA = {"User-Agent": "Mozilla/5.0 (compatible; research bot)"}


def generate_month_range(start, end):
    s = datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end, "%Y-%m-%d")
    months = []
    y, m = s.year, s.month
    while (y, m) <= (e.year, e.month):
        months.append((y, m))
        m += 1
        if m > 12:
            m, y = 1, y + 1
    return months


def fetch_monthly_sitemap(year, month):
    url = f"https://www.wsj.com/sitemaps/web/wsj/en/sitemap_wsj_en_m{month}_{year}.xml"
    try:
        r = requests.get(url, headers=SITEMAP_UA, timeout=15)
        if r.status_code != 200:
            return []
        root = ET.fromstring(r.text)
        return [
            {
                "url": e.find("s:loc", NS).text,
                "lastmod": (e.find("s:lastmod", NS).text
                            if e.find("s:lastmod", NS) is not None else None),
            }
            for e in root.findall(".//s:url", NS)
            if e.find("s:loc", NS) is not None
        ]
    except Exception as ex:
        print(f"  Error {year}-{month:02d}: {ex}")
        return []


def slug_from_url(url):
    path = url.split("wsj.com/")[-1]
    slug = re.sub(r"-[0-9a-f]{8}$", "", path.split("/")[-1])
    return slug.replace("-", " ")


def matches_keywords(url, keywords):
    slug = slug_from_url(url).lower()
    return any(kw.lower() in slug for kw in keywords)

In [ ]:
# ── Run Phase 1 (or load cached results) ─────────────────────

FORCE_RECRAWL = False  # Set True to ignore cached URL list

if os.path.exists(URL_LIST_PATH) and not FORCE_RECRAWL:
    url_df = pd.read_csv(URL_LIST_PATH)
    print(f"Loaded {len(url_df):,} cached URLs from {URL_LIST_PATH}")
else:
    months = generate_month_range(CONFIG["start_date"], CONFIG["end_date"])
    print(f"Crawling {len(months)} monthly sitemaps "
          f"({months[0][0]}-{months[0][1]:02d} to {months[-1][0]}-{months[-1][1]:02d})...\n")

    all_articles = []
    for year, month in months:
        print(f"  {year}-{month:02d}...", end=" ")
        articles = fetch_monthly_sitemap(year, month)
        matched = [a for a in articles if matches_keywords(a["url"], CONFIG["keywords"])]
        all_articles.extend(matched)
        print(f"{len(articles)} total, {len(matched)} matched "
              f"(running: {len(all_articles):,})")
        time.sleep(0.3)

    url_df = pd.DataFrame(all_articles)
    url_df["title"] = url_df["url"].apply(lambda u: slug_from_url(u).title())
    url_df.drop_duplicates(subset=["url"], inplace=True)
    url_df.to_csv(URL_LIST_PATH, index=False)
    print(f"\nSaved {len(url_df):,} URLs to {URL_LIST_PATH}")

# Stats
print(f"\nTotal URLs: {len(url_df):,}")
if "lastmod" in url_df.columns:
    url_df["_year"] = pd.to_datetime(url_df["lastmod"], errors="coerce").dt.year
    print("\nArticles per year:")
    print(url_df["_year"].value_counts().sort_index().to_string())
    url_df.drop(columns=["_year"], inplace=True)

Loaded 26,989 cached URLs from /content/drive/MyDrive/wsj_data/wsj_urls.csv

Total URLs: 26,989

Articles per year:
_year
2013      17
2014    1946
2015    3234
2016    3368
2017    2486
2018    2435
2019    2203
2020    2009
2021    1753
2022    2423
2023    1654
2024    1387
2025    1406
2026     668


---
## Phase 2: Authentication

Two modes:
1. **Auto-login** — HTTP-based OAuth2 flow (same as v1). Works unless CAPTCHA is active.
2. **Manual cookie injection** — You log in via your browser, copy the cookies, paste here. Bypasses CAPTCHA entirely.

The notebook tries auto-login first. If it fails (CAPTCHA), it falls back to manual injection with step-by-step instructions.

In [ ]:
def login_wsj(email, password):
    """Log into WSJ via Dow Jones SSO and return an authenticated curl_cffi Session."""
    session = cffi_requests.Session(impersonate="chrome")

    # Step 1: Initiate OAuth flow
    print("  Initiating login flow...")
    r = session.get("https://www.wsj.com/login", timeout=20)
    final_url = r.url

    if not any(s in final_url for s in ("login-page", "/login", "authorize")):
        print(f"  Unexpected redirect: {final_url}")
        return None

    if "authorize" in final_url and "login-page" not in final_url:
        query = urlparse(final_url).query
        login_page_url = f"https://sso.accounts.dowjones.com/login-page?{query}"
        print("  Navigating to /login-page...")
        r = session.get(login_page_url, timeout=20)
        final_url = r.url

    params = parse_qs(urlparse(final_url).query)
    client_id = (params.get("client_id") or params.get("client", [None]))[0]
    state = params.get("state", [None])[0]
    nonce = params.get("nonce", [None])[0]
    scope = params.get("scope", [None])[0]
    redirect_uri = params.get("redirect_uri", [None])[0]
    ui_locales = params.get("ui_locales", [None])[0]
    response_type = params.get("response_type", ["code"])[0]

    # Step 2: Extract CSRF token
    m = re.search(r"AUTH_CONFIG\s*=\s*'([^']+)'", r.text)
    if not m:
        if "captcha" in r.text.lower() or "geo.captcha" in r.text:
            print("  CAPTCHA detected — auto-login blocked.")
        else:
            title_m = re.search(r'<title>(.+?)</title>', r.text)
            page_title = title_m.group(1) if title_m else "unknown"
            print(f"  Could not find AUTH_CONFIG (page: '{page_title}')")
        return None

    config = json.loads(base64.b64decode(m.group(1)))
    csrf = config["extraParams"]["_csrf"]
    print(f"  Got OAuth params (client_id={client_id[:8]}...)")

    # Step 3: POST /authenticate
    print("  Authenticating...")
    r = session.post(
        "https://sso.accounts.dowjones.com/authenticate",
        json={
            "username": email, "password": password, "state": state,
            "client_id": client_id, "csrf": csrf, "response_mode": None,
            "scope": scope,
            "code_challenge": params.get("code_challenge", [None])[0],
            "realm": "DJldap",
            "code_challenge_method": params.get("code_challenge_method", [None])[0],
            "nonce": nonce, "ui_locales": ui_locales,
            "redirect_uri": redirect_uri, "response_type": response_type,
        },
        headers={
            "Accept": "application/json", "Content-Type": "application/json",
            "X-REQUEST-SCHEME": "https", "X-REMOTE-USER": email,
            "X-REQUEST-EDITIONID": "wsj", "Referer": final_url,
        },
        timeout=20,
    )

    if r.status_code == 401:
        print("  Wrong username or password"); return None
    if r.status_code == 403:
        print("  Account locked or rate-limited"); return None
    if r.status_code != 200:
        print(f"  HTTP {r.status_code}: {r.text[:200]}"); return None

    # Step 4: Complete session setup
    content_type = r.headers.get("content-type", "")
    if "application/json" in content_type:
        data = r.json()
        redirect_url = data.get("redirect_uri") or data.get("url")
        if redirect_url:
            print("  Completing session (JSON redirect)...")
            session.get(redirect_url, timeout=20)
        else:
            print(f"  No redirect URL in JSON"); return None
    else:
        form_action = re.search(r'<form[^>]*action="([^"]+)"', r.text)
        if not form_action:
            print("  Could not parse callback form"); return None
        callback_url = form_action.group(1).replace("&amp;", "&")
        hidden = dict(re.findall(r'<input[^>]*name="([^"]+)"[^>]*value="([^"]*)"', r.text))
        print("  Completing session (form callback)...")
        session.post(callback_url, data=hidden, timeout=20)

    n = len(list(session.cookies.jar))
    print(f"  Login successful! {n} cookies.")
    return session

In [ ]:
def _session_from_cookie_string(cookie_str):
    """Create a curl_cffi session from a browser cookie string."""
    s = cffi_requests.Session(impersonate="chrome")
    for pair in cookie_str.split('; '):
        if '=' in pair:
            name, value = pair.split('=', 1)
            s.cookies.set(name.strip(), value.strip(), domain=".wsj.com")
    n = sum(1 for _ in s.cookies)
    print(f"  Injected {n} cookies.")
    return s


def get_auth_session(email, password, auto_login_count=0):
    """Get a single authenticated session. Tries auto-login, falls back to manual cookies.

    Returns (session, auth_time, new_login_count).
    """
    if auto_login_count < CONFIG["max_auto_logins"]:
        print(f"[Auth] Auto-login (attempt {auto_login_count + 1}/{CONFIG['max_auto_logins']})...")
        session = login_wsj(email, password)
        if session is not None:
            return session, time.time(), auto_login_count + 1
        print("\n[Auth] Auto-login failed. Falling back to manual cookie injection.\n")
    else:
        print(f"\n[Auth] Reached {auto_login_count} auto-logins — using manual cookies.\n")

    # Manual cookie injection
    print("=" * 60)
    print("MANUAL COOKIE INJECTION")
    print("=" * 60)
    print()
    print("1. Open https://www.wsj.com in your browser")
    print("2. Log in (solve CAPTCHA manually if needed)")
    print("3. Open DevTools (F12) > Network tab")
    print("4. Refresh the page (Ctrl+R)")
    print("5. Click the first request to 'www.wsj.com'")
    print("6. Under Request Headers, find 'Cookie:'")
    print("7. Right-click the Cookie VALUE > Copy value")
    print("8. Paste below (it will be a long string)")
    print()
    cookie_str = input("Paste cookie string: ").strip()
    if not cookie_str:
        raise ValueError("No cookies provided.")
    session = _session_from_cookie_string(cookie_str)
    return session, time.time(), auto_login_count


# ── Prompt for credentials and authenticate ──────────────────

email = input("WSJ email: ")
password = getpass("WSJ password: ")
login_count = 0

session, auth_time, login_count = get_auth_session(email, password, login_count)
print(f"\nAuthenticated at {datetime.fromtimestamp(auth_time).strftime('%H:%M:%S')}")
print(f"Auto-logins used: {login_count}/{CONFIG['max_auto_logins']}")

WSJ email: journals@yonsei.ac.kr
WSJ password: ··········
[Auth] Auto-login (attempt 1/4)...
  Initiating login flow...
  Got OAuth params (client_id=5hssEAdM...)
  Authenticating...
  Completing session (form callback)...
  Login successful! 12 cookies.

Authenticated at 10:07:11
Auto-logins used: 1/4


---
## Phase 2: Article Fetching

Fetches articles in **batches** sized to fit within one session window (~20 min). Between batches, refreshes the session (auto-login or manual cookies). All progress is checkpointed to Google Drive.

**Resume:** Just re-run — the checkpoint skips already-fetched articles automatically.

In [ ]:
# ── Article extraction ────────────────────────────────────────

def extract_text_from_jsonld(html):
    match = re.search(
        r'<script[^>]*type="application/ld\+json"[^>]*>(.*?)</script>',
        html, re.DOTALL,
    )
    if not match:
        return None
    try:
        data = json.loads(match.group(1))
        if isinstance(data, list):
            data = next(
                (d for d in data if d.get("@type") in ("NewsArticle", "Article")),
                data[0],
            )
        return data.get("articleBody") or data.get("description")
    except (json.JSONDecodeError, IndexError, StopIteration):
        return None


def fetch_wsj_article(url, session):
    """Fetch one WSJ article. Returns (text, title, status)."""
    try:
        r = session.get(url, timeout=20)
        if r.status_code == 403:
            return None, None, "403_blocked"
        if r.status_code == 401:
            return None, None, "401_cookies_expired"
        if r.status_code != 200:
            return None, None, f"http_{r.status_code}"

        html = r.text

        # Title
        tm = re.search(r'<title[^>]*>(.+?)</title>', html, re.DOTALL)
        title = re.sub(r'\s*[-|]\s*WSJ$', '', tm.group(1).strip()) if tm else None

        # Paywall check
        if '"content_type":"paywall"' in html or 'class="wsj-snippet-body"' in html:
            return None, title, "paywall_hit"

        # Primary: trafilatura
        text = trafilatura.extract(
            html, include_comments=False, include_tables=False, favor_recall=True,
        )
        # Fallback: JSON-LD
        if not text or len(text) < 200:
            jt = extract_text_from_jsonld(html)
            if jt and (not text or len(jt) > len(text)):
                text = jt

        if text and len(text.split()) >= 100:
            return text, title, "ok"
        return text, title, "too_short"
    except Exception as e:
        return None, None, f"error: {e}"

In [ ]:
# ── Checkpoint I/O ────────────────────────────────────────────

CP_COLS = ["url", "lastmod", "title", "full_text", "fetch_status"]
DONE_STATUSES = {"ok", "too_short"}  # Don't retry these


def load_done_urls(path):
    """Return set of URLs that are permanently done (ok or too_short)."""
    if not os.path.exists(path):
        return set()
    try:
        df = pd.read_csv(path, usecols=["url", "fetch_status"], on_bad_lines="skip")
        return set(df[df["fetch_status"].isin(DONE_STATUSES)]["url"])
    except Exception:
        return set()


def append_checkpoint(rows, path):
    """Append result rows to checkpoint CSV."""
    if not rows:
        return
    df = pd.DataFrame(rows, columns=CP_COLS)
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

In [ ]:
# ── Sequential fetcher ────────────────────────────────────────

def fetch_sequential(url_records, session, auth_time, checkpoint_path):
    """Fetch articles one at a time with human-like delays.

    Checkpoints every CONFIG['checkpoint_every'] articles.
    Returns early on session expiry or repeated auth failures.

    Returns (ok_count, processed_count, stop_reason).
    stop_reason: "done" | "session_expired" | "auth_failed"
    """
    total = len(url_records)
    buf = []
    ok = 0
    fail = 0
    consec_auth = 0
    start = time.time()
    dmin, dmax = CONFIG["delay_range"]
    cp_every = CONFIG["checkpoint_every"]

    for i, item in enumerate(url_records):
        # Proactive session expiry check
        session_age = (time.time() - auth_time) / 60
        if session_age >= CONFIG["session_ttl_minutes"]:
            if buf:
                append_checkpoint(buf, checkpoint_path)
                buf = []
            print(f"\n  Session TTL reached ({session_age:.0f}min). Pausing for re-auth.")
            return ok, i, "session_expired"

        url = item["url"]
        text, title, status = fetch_wsj_article(url, session)

        buf.append([
            url,
            item.get("lastmod"),
            title or item.get("title", ""),
            text,
            status,
        ])

        if status == "ok":
            ok += 1
            consec_auth = 0
        elif status in ("401_cookies_expired", "paywall_hit"):
            fail += 1
            consec_auth += 1
            if consec_auth >= 5:
                if buf:
                    append_checkpoint(buf, checkpoint_path)
                    buf = []
                print(f"\n  5 consecutive auth failures. Session is dead.")
                return ok, i + 1, "auth_failed"
        else:
            fail += 1
            consec_auth = 0

        # Flush to disk
        if len(buf) >= cp_every:
            append_checkpoint(buf, checkpoint_path)
            buf = []

        # Progress log every 50 articles
        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed
            eta_h = (total - i - 1) / rate / 3600
            print(f"  [{i+1:,}/{total:,}] {ok:,} ok, {fail:,} fail | "
                  f"{rate:.1f} art/s | ETA {eta_h:.1f}h")

        # Human-like delay with occasional longer pause
        delay = random.uniform(dmin, dmax)
        if random.random() < 0.005:  # ~0.5% chance → ~1 pause per 200 articles
            pause = random.uniform(10, 30)
            print(f"  (random pause {pause:.0f}s)")
            delay += pause
        time.sleep(delay)

    # Flush remaining
    if buf:
        append_checkpoint(buf, checkpoint_path)

    elapsed = time.time() - start
    rate = total / elapsed if elapsed > 0 else 0
    print(f"\n  Chunk complete: {ok:,} ok, {fail:,} fail "
          f"in {elapsed/3600:.1f}h ({rate:.1f} art/s)")
    return ok, total, "done"

---
## Verification: Quick Auth Test

In [ ]:
# ── Verify authentication with 3 articles ─────────────────────

print("Verifying authentication with 3 articles...\n")
test_items = url_df.head(3).to_dict('records')
v_ok = 0

for i, item in enumerate(test_items):
    url = item["url"]
    print(f"[{i+1}/3] {slug_from_url(url).title()}")
    text, title, status = fetch_wsj_article(url, session)
    if status == "ok":
        v_ok += 1
        wc = len(text.split())
        print(f"  OK: '{title}' ({wc} words)")
        print(f"  Preview: {text[:200]}...\n")
    else:
        print(f"  FAILED: {status}\n")
    time.sleep(2)

if v_ok == 0:
    print("WARNING: All 3 failed. Your IP may be blocked by Akamai.")
    print("Try: Runtime > Disconnect and delete runtime > Reconnect (new IP)")
    print("Or: re-run the auth cell with manual cookie injection.")
elif v_ok < 3:
    print(f"Partial success ({v_ok}/3). Some articles may be non-standard pages.")
else:
    print("All 3 verified. Ready to proceed.")

Verifying authentication with 3 articles...

[1/3] Djfdbr0020130124E91Oilke5
  FAILED: 401_cookies_expired

[2/3] Tyson Cole S Romanesco With Seared Broccolini Lemon Curd And Quinoa Slow Food Fast 11595929152
  FAILED: 401_cookies_expired

[3/3] Seafood Restaurant Meets The Gulf Coast 11595584528
  FAILED: 401_cookies_expired

Try: Runtime > Disconnect and delete runtime > Reconnect (new IP)
Or: re-run the auth cell with manual cookie injection.


---
## Full Production Run

Fetches all remaining articles sequentially. Re-authenticates when the session expires (~every 20 min). All progress is checkpointed to Google Drive.

**To resume after a disconnect:** Restart runtime, re-run all cells from the top. Phase 1 loads cached URLs. This cell loads the checkpoint and picks up where it left off — no work is repeated.

In [ ]:
# ── Full production run ───────────────────────────────────────

# Load all URLs and shuffle to avoid sequential-access bot signature
all_records = url_df.to_dict('records')
random.seed(42)  # Deterministic shuffle so resume doesn't re-order
random.shuffle(all_records)

done_urls = load_done_urls(CHECKPOINT_PATH)
remaining = [r for r in all_records if r["url"] not in done_urls]

print(f"Total articles:  {len(all_records):,}")
print(f"Already done:    {len(done_urls):,}")
print(f"Remaining:       {len(remaining):,}")
print()

run_start = time.time()
total_ok = 0
round_num = 0

while remaining:
    round_num += 1

    # Re-authenticate if session is stale
    session_age = (time.time() - auth_time) / 60
    if round_num > 1 or session_age >= CONFIG["session_ttl_minutes"]:
        print(f"\n--- Re-authenticating (session age: {session_age:.0f}min) ---")
        session, auth_time, login_count = get_auth_session(
            email, password, login_count,
        )

    print(f"\n{'='*60}")
    print(f"ROUND {round_num}: {len(remaining):,} articles remaining")
    print(f"{'='*60}\n")

    ok, processed, reason = fetch_sequential(
        remaining, session, auth_time, CHECKPOINT_PATH,
    )
    total_ok += ok

    # Reload remaining from checkpoint
    done_urls = load_done_urls(CHECKPOINT_PATH)
    remaining = [r for r in all_records if r["url"] not in done_urls]

    if reason == "session_expired":
        print(f"Session expired after {processed:,} articles. Re-authenticating...")
        continue

    if reason == "auth_failed":
        print(f"\nAuth failed after {processed:,} articles.")
        print("Will attempt re-login for next round...")
        # Force re-auth on next iteration by resetting auth_time
        auth_time = 0
        continue

    # reason == "done"
    break

total_elapsed = time.time() - run_start
print(f"\n{'='*60}")
print(f"ALL DONE")
print(f"{'='*60}")
print(f"Total fetched OK: {total_ok:,}")
print(f"Total time: {total_elapsed/3600:.1f} hours")
print(f"Auto-logins used: {login_count}")

Total articles:  26,989
Already done:    7,019
Remaining:       19,970


ROUND 1: 19,970 articles remaining

  [50/19,970] 45 ok, 5 fail | 0.4 art/s | ETA 14.4h
  (random pause 23s)
  (random pause 29s)
  [100/19,970] 93 ok, 7 fail | 0.3 art/s | ETA 17.3h
  (random pause 13s)
  [150/19,970] 143 ok, 7 fail | 0.3 art/s | ETA 17.0h
  [200/19,970] 191 ok, 9 fail | 0.3 art/s | ETA 16.2h
  [250/19,970] 239 ok, 11 fail | 0.3 art/s | ETA 15.8h
  [300/19,970] 287 ok, 13 fail | 0.4 art/s | ETA 15.5h
  [350/19,970] 333 ok, 17 fail | 0.4 art/s | ETA 15.2h
  [400/19,970] 381 ok, 19 fail | 0.4 art/s | ETA 15.0h

  Session TTL reached (20min). Pausing for re-auth.
Session expired after 432 articles. Re-authenticating...

--- Re-authenticating (session age: 20min) ---
[Auth] Auto-login (attempt 2/4)...
  Initiating login flow...
  Got OAuth params (client_id=5hssEAdM...)
  Authenticating...
  Completing session (form callback)...
  Login successful! 12 cookies.

ROUND 2: 19,538 articles remaining

  [

KeyboardInterrupt: Interrupted by user

---
## Results & Export

In [ ]:
# ── Load checkpoint and produce final dataset ─────────────────

df = pd.read_csv(CHECKPOINT_PATH)
print(f"Total rows in checkpoint: {len(df):,}")
print(f"\nStatus breakdown:")
print(df["fetch_status"].value_counts().to_string())

# Keep successful extractions only
results = df[df["fetch_status"] == "ok"].copy()
results.drop(columns=["fetch_status"], inplace=True)
results.drop_duplicates(subset=["url"], inplace=True)
results["word_count"] = results["full_text"].str.split().str.len()

print(f"\nFinal dataset: {len(results):,} articles")
print(f"\nWord count stats:")
print(results["word_count"].describe().to_string())

# Save
results.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

# Download
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except ImportError:
    print("Not in Colab — file saved to disk.")